<a href="https://colab.research.google.com/github/23SCSE1012097/Intelligent-Resume-Screening-and-Job-Recommendation-System/blob/main/Intelligent_Resume_Screening_and_Job_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers pypdf nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 10.2 MB/s eta 0:00:00


In [2]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.corpus import stopwords
from nltk import word_tokenize, sent_tokenize

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

STOP_WORDS = set(stopwords.words("english"))
print("Libraries ready")

Libraries ready


In [3]:
from google.colab import files
uploaded = files.upload()

Saving ajayresume-1.pdf to ajayresume-1.pdf


In [6]:
import pypdf
import io

# Get the PDF filename and content from the uploaded dictionary
pdf_filename = list(uploaded.keys())[0]
pdf_content = uploaded[pdf_filename]

# Open the PDF file from its byte content
pdf_file = io.BytesIO(pdf_content)
pdf_reader = pypdf.PdfReader(pdf_file)

resume_text = ""
for page_num in range(len(pdf_reader.pages)):
    page = pdf_reader.pages[page_num]
    resume_text += page.extract_text() + "\n"

# Create a DataFrame with the extracted text
# Assuming for a single uploaded PDF, we can assign a default ID and Category
resume_df = pd.DataFrame([
    {
        "ID": 1, # Placeholder ID
        "Category": "Uploaded_Resume", # Placeholder Category
        "resume_text": resume_text
    }
]).dropna(subset=["resume_text"]).reset_index(drop=True)

print(resume_df.shape)
display(resume_df.head())

(1, 3)


,ID,Category,resume_text
0,1,Uploaded_Resume,AJAY KUMAR\nPortfolio — ajaykumar665748@gmail....


In [9]:
from google.colab import files
print("Please upload your job descriptions CSV file (e.g., job_descriptions.csv).")
uploaded_jobs = files.upload()

Please upload your job descriptions CSV file (e.g., job_descriptions.csv).


Saving Galgotias University Mail - Notice For Virtual Campus Drive 2027 - Amantya Technologies (1).pdf to Galgotias University Mail - Notice For Virtual Campus Drive 2027 - Amantya Technologies (1).pdf


In [12]:
import io
import pypdf

job_filename = list(uploaded_jobs.keys())[0]
job_content = uploaded_jobs[job_filename]

# Open the PDF file from its byte content
pdf_file = io.BytesIO(job_content)
pdf_reader = pypdf.PdfReader(pdf_file)

job_text = ""
for page_num in range(len(pdf_reader.pages)):
    page = pdf_reader.pages[page_num]
    job_text += page.extract_text() + "\n"

# Create a DataFrame with the extracted text
# Assuming for a single uploaded PDF, we can assign a default title and ID
job_df = pd.DataFrame([
    {
        "job_id": 1, # Placeholder ID
        "job_title": "Uploaded_Job_Description", # Placeholder title
        "job_text": job_text
    }
]).dropna(subset=["job_text"]).reset_index(drop=True)

print(job_df.shape)
display(job_df.head())

(1, 3)


,job_id,job_title,job_text
0,1,Uploaded_Job_Description,Dr. S. Premkumar <s.premkumar@galgotiasunivers...


In [14]:
# The job description data was loaded from the uploaded PDF in the previous step.
# Displaying the head of the DataFrame to confirm.
print(job_df.shape)
display(job_df.head())

(1, 3)


,job_id,job_title,job_text
0,1,Uploaded_Job_Description,Dr. S. Premkumar <s.premkumar@galgotiasunivers...


In [15]:
def clean_text(text):
    # Convert to lowercase
    text = str(text).lower()
    # Remove numbers and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize and remove stopwords
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in STOP_WORDS]
    return " ".join(tokens)

resume_df["cleaned_resume_text"] = resume_df["resume_text"].apply(clean_text)
job_df["cleaned_job_text"] = job_df["job_text"].apply(clean_text)

print("Cleaned Resume Text Sample:")
display(resume_df[['resume_text', 'cleaned_resume_text']].head())
print("\nCleaned Job Description Text Sample:")
display(job_df[['job_text', 'cleaned_job_text']].head())

Cleaned Resume Text Sample:


,resume_text,cleaned_resume_text
0,AJAY KUMAR\nPortfolio — ajaykumar665748@gmail....,ajay kumar portfolio ajaykumargmailcom github ...



Cleaned Job Description Text Sample:


,job_text,cleaned_job_text
0,Dr. S. Premkumar <s.premkumar@galgotiasunivers...,dr premkumar spremkumargalgotiasuniversityedui...


In [16]:
print("Loading Sentence Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded.")

Loading Sentence Transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.


In [17]:
print("Generating embeddings for resume and job description...")
resume_embeddings = model.encode(resume_df['cleaned_resume_text'].tolist())
job_embeddings = model.encode(job_df['cleaned_job_text'].tolist())

print(f"Resume embeddings shape: {resume_embeddings.shape}")
print(f"Job description embeddings shape: {job_embeddings.shape}")

Generating embeddings for resume and job description...
Resume embeddings shape: (1, 384)
Job description embeddings shape: (1, 384)


In [18]:
print("Calculating cosine similarity...")
# Calculate cosine similarity between the resume and job description
# Assuming one resume and one job description for now
similarity_score = cosine_similarity(resume_embeddings, job_embeddings)[0][0]

print(f"Resume-Job Description Matching Score (Cosine Similarity): {similarity_score:.4f}")

Calculating cosine similarity...
Resume-Job Description Matching Score (Cosine Similarity): 0.6959


### Matching Score Interpretation

The cosine similarity score ranges from -1 to 1:
- **1**: Indicates identical content.
- **0**: Indicates no similarity.
- **-1**: Indicates opposite content.

A higher score means a better match between your resume and the job description. You can set a threshold (e.g., 0.5 or 0.7) to determine what constitutes a 'good' match for your specific use case.